# 2.4 — Do raw às janelas e features (Forma 2, CSV)

Aqui está o **ponto forte da Forma 2**: como guardamos o sinal bruto, nós mesmos definimos a
janela no Colab — podemos **descartar a transição** e recortar o trecho exato do fenômeno
antes de janelar (a Forma 1 já chegou janelada do edge, sem essa liberdade).

Geramos as **mesmas 7 features** do edge (`mean_ax/ay/az`, `std_ax/ay/az`, `rms_mag`) por
**janela fixa de 100 amostras (1 s) sem sobreposição** e salvamos um CSV de features para o
notebook **2.5**.

> **Atende Sprint 4 – item 1:** features por janela, dataset balanceado (≥30 por classe).

In [ ]:
!pip install -q pandas numpy

## Carregar o raw

In [ ]:
import pandas as pd, numpy as np

try:
    from google.colab import files
    enviados = files.upload()
    arquivos = list(enviados.keys())
except Exception:
    import glob
    arquivos = glob.glob("coleta_*.csv")

raw = pd.concat([pd.read_csv(a) for a in arquivos], ignore_index=True)
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
print(raw["label"].value_counts())

## Descartar a transição (Aula 14, slide 22)

Os primeiros instantes de cada rodada costumam ter ruído (mão ajustando o sensor). Descartamos
as primeiras `DESCARTE_S` segundos de cada classe. Ajuste à vontade — essa flexibilidade é
exatamente o que a Forma 2 oferece.

In [ ]:
FS = 100               # Hz
DESCARTE_S = 3         # segundos descartados no inicio de cada classe

partes = []
for lab, g in raw.groupby("label"):
    g = g.sort_values("timestamp").iloc[DESCARTE_S * FS:]
    partes.append(g)
raw_limpo = pd.concat(partes, ignore_index=True)
print("Amostras após descarte:")
print(raw_limpo["label"].value_counts())

## Janela fixa (100 amostras, sem sobreposição) → 7 features

As fórmulas são as mesmas do firmware (`std`/`rms` com divisão por N — `np.std` usa `ddof=0`,
que bate com o `calcStd` do ESP32).

In [ ]:
TAMANHO_JANELA = 100   # 100 amostras @ 100 Hz = 1 s

def features_da_janela(j):
    ax, ay, az = j["ax"].values, j["ay"].values, j["az"].values
    rms_mag = np.sqrt(np.mean(ax**2 + ay**2 + az**2))
    return {
        "mean_ax": ax.mean(), "mean_ay": ay.mean(), "mean_az": az.mean(),
        "std_ax":  ax.std(),  "std_ay":  ay.std(),  "std_az":  az.std(),
        "rms_mag": rms_mag,
    }

linhas = []
for lab, g in raw_limpo.groupby("label"):
    g = g.reset_index(drop=True)
    n_janelas = len(g) // TAMANHO_JANELA
    for k in range(n_janelas):
        janela = g.iloc[k*TAMANHO_JANELA:(k+1)*TAMANHO_JANELA]
        feat = features_da_janela(janela)
        feat["label"] = lab
        feat["ordem"] = k          # posicao temporal da janela (usada no split)
        linhas.append(feat)

feat_df = pd.DataFrame(linhas)
print("Janelas por classe:")
print(feat_df["label"].value_counts())
assert (feat_df["label"].value_counts() >= 30).all(), "Menos de 30 janelas em alguma classe — colete mais!"
feat_df.head()

## Split treino/teste SEM vazamento

Janelas vizinhas no tempo são quase iguais — um split aleatório vazaria informação
(Aula 14, slide 25). Fazemos um **split cronológico por classe** (primeiras 70% das janelas →
treino; últimas 30% → teste), usando a coluna `ordem`. O modelo de fato fica no notebook 2.5;
aqui só salvamos as features.

In [ ]:
feat_df["split"] = ""
for lab, g in feat_df.groupby("label"):
    corte = int(len(g) * 0.7)
    feat_df.loc[g.index[:corte], "split"] = "treino"
    feat_df.loc[g.index[corte:], "split"] = "teste"

feat_df.to_csv("features_from_raw.csv", index=False)
print(feat_df.groupby(["label", "split"]).size())
try:
    from google.colab import files
    files.download("features_from_raw.csv")
except Exception:
    print("Salvo em features_from_raw.csv")

## Comparativo raw × features (entrega da Sprint 4)

- **Raw:** 1 linha = 1 leitura (~6000/classe/min). Útil para visualizar e re-janelar.
- **Features:** 1 linha = 1 janela de 1 s (as 7 colunas acima). É o que vai para o modelo.

Leve o `features_from_raw.csv` para o notebook **2.5** (mesmo Random Forest da Forma 1).